# 📌 Rate-Distortion Forget

![Topic](https://img.shields.io/badge/Topic-RD_Forget-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-RAG-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-September%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — RD-Forget lets an AI agent keep every fact it has ever learned, but only show the LLM the facts that are actually relevant and still valid for the current question. Instead of deleting old information, it separates "what's stored" from "what's used," so an outdated fact can be hidden from a current-status question but still surface for a historical one (Li & Li, 2026).</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

Most long-running LLM agents just keep appending observations to a growing history and either dump the whole thing into context or retrieve loosely by similarity. This creates a hard tradeoff: keep everything and risk an old, superseded fact (an old employer, an old address) confusing the model's answer, or aggressively prune history and lose facts a later question might need. 

RD-Forget argues the fix isn't to choose one point on that tradeoff, but to stop treating storage and use as the same decision (Li & Li, 2026). 

It keeps a full, untouched archive of every observation, and separately builds a small, question-specific "view" of that archive for each new query. Forgetting becomes local to a question: a fact can be excluded from one answer while remaining fully retrievable for a different question later, using the same underlying record.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

### 2.1 Source archive
Every observation the agent has ever received is stored, untouched, indexed by things like session ID or timestamp (Li & Li, 2026).

### 2.2 Curation
A frozen LLM "curator" reads the current question plus the archive and pulls out only the facts relevant to answering it. It also assigns each fact a slot, written as subject | relation | scope (e.g. "Mira | employer | current"), so that facts about the same relationship are grouped together (Li & Li, 2026).

### 2.3 Supersession (the actual "forgetting" step)
If two facts share a slot and one is more recent, the older one is marked SUPERSEDED and linked to the fact that replaced it. It isn't deleted, it's just tagged as inactive for this slot (Li & Li, 2026).

### 2.4 Eligibility / rescue
For a normal/current-state question, only active (non-superseded) facts are eligible. But if the curator detects the query is a historical or "how did this change over time" question, it re-admits the superseded facts too, a mechanism the paper calls intent-aware retrieval or "rescue" (Li & Li, 2026).

### 2.5 Ranking and packing
Eligible facts are ranked (mostly by overlap with the question, with a bonus for active facts), then a second LLM call selects and orders them, and a greedy packer fits as many as possible into a fixed token budget. The paper frames this as a rate-distortion problem: pick the subset of facts that best preserves answer quality without exceeding the budget (Li & Li, 2026).

### 2.6 Answering
The frozen LLM answers the question using only this packed, query-specific view, not the raw history.

![RD_Forget](../assets/RD_Forget.png)



---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Best accuracy in every test** | RD-Forget scored highest in all twelve model/benchmark combinations against ACE and ReasoningBank baselines, with three-suite mean accuracy ranging from 66.59% (Qwen) to 86.71% (Luna) depending on the model (Li & Li, 2026). |
| 🟢 | **Big wins on fact consolidation specifically** | On tasks that require tracking which fact is current versus outdated, RD-Forget beat the better baseline by 11.00 to 26.00 percentage points across four different LLM backbones (Li & Li, 2026). |
| 🟢 | **Old facts aren't lost, just hidden** | Because superseded facts stay in the archive and only lose eligibility for the current view, a later historical query can recover them, something a standard agent that overwrites or prunes memory cannot do. |
| 🟢 | **Forgetting works at the level of a single relation** | Slot-based supersession lets one outdated fact (an old employer) get suppressed while a related fact in a different slot (that employer's location) stays available, avoiding the all-or-nothing pruning typical history-conditioned agents rely on. |
| 🟢 | **Handles both current-state and historical questions well** | The query-intent evaluation showed RD-Forget leading in all eight model/benchmark combinations on BEAM and PersonaMem, with gains of 8.47 to 20.72 points over the better-performing baseline (Li & Li, 2026). |
| 🔴 | **Extra LLM calls per query** | Curation, selection, and answering are each separate frozen-LLM calls, adding latency and token cost a plain retrieve-and-answer agent avoids (Li & Li, 2026). |
| 🔴 | **Accuracy leans heavily on the forgetting step working correctly** | The ablation that disabled forgetting entirely produced the largest score drop of any variant, up to 33.33 points lower than the full method on one benchmark, showing how much the result depends on this one mechanism (Li & Li, 2026). |
| 🔴 | **Detecting "is this a historical question" is imperfect** | The rescue mechanism that re-admits old facts relies on a signal built from lexical cues and task-type labels rather than a clean, generalizable classifier, which is more fragile than an agent that just always shows full history (Li & Li, 2026). |
| 🔴 | **Evaluation isn't fully independent** | Curation, selection, answering, and judging all use the same model family in each run, so the paper itself frames its scores as within-protocol rather than judged by a fully separate evaluator (Li & Li, 2026). |
| 🔴 | **Scope judgments can still be fuzzy** | Deciding whether a new fact truly replaces an old one, versus just adding a related detail, still depends on the curator's judgment calls, something a raw-history agent sidesteps entirely by not making that distinction at all. |


---
## 4. Code Example

> **Goal:** This is a deliberately simplified illustration of the idea, not the paper's actual system (no LLM calls, no rate-distortion optimization, no real ranking model).

In [1]:
"""
Toy illustration of RD-Forget's core idea:
- keep every fact ever seen (the archive)
- group facts into "slots"
- mark older same-slot facts as SUPERSEDED, not deleted
- build a query-specific view depending on whether the query is
  "current" or "historical"
"""

from dataclasses import dataclass, field
from typing import List

@dataclass
class Fact:
    text: str
    slot: str          # e.g. "mira|employer"
    status: str = "ACTIVE"   # ACTIVE or SUPERSEDED
    superseded_by: str = None

class SourceArchive:
    def __init__(self):
        self.facts: List[Fact] = []

    def add(self, fact: Fact):
        # Same-slot supersession: the newest fact in a slot
        # deactivates the previous one, but nothing is deleted.
        for old in self.facts:
            if old.slot == fact.slot and old.status == "ACTIVE":
                old.status = "SUPERSEDED"
                old.superseded_by = fact.text
        self.facts.append(fact)

    def build_view(self, query_is_historical: bool) -> List[Fact]:
        if query_is_historical:
            # "rescue": superseded facts become eligible again
            return self.facts
        # current-state query: only active facts are shown
        return [f for f in self.facts if f.status == "ACTIVE"]


# --- demo ---
archive = SourceArchive()
archive.add(Fact("Mira works at Acme.", slot="mira|employer"))
archive.add(Fact("Beryl is located in Lyon.", slot="beryl|location"))
archive.add(Fact("Mira now works at Beryl.", slot="mira|employer"))

print("Current-state view (used to answer 'Where does Mira work now?'):")
for f in archive.build_view(query_is_historical=False):
    print(" -", f.text, f"[{f.status}]")

print("\nHistorical view (used to answer 'Where did Mira used to work?'):")
for f in archive.build_view(query_is_historical=True):
    print(" -", f.text, f"[{f.status}]")

Current-state view (used to answer 'Where does Mira work now?'):
 - Beryl is located in Lyon. [ACTIVE]
 - Mira now works at Beryl. [ACTIVE]

Historical view (used to answer 'Where did Mira used to work?'):
 - Mira works at Acme. [SUPERSEDED]
 - Beryl is located in Lyon. [ACTIVE]
 - Mira now works at Beryl. [ACTIVE]


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **RD-Forget separates what an agent stores from what it uses at answer time.**
- **Forgetting means hiding a fact from a specific query, not deleting it from memory.**
- **Facts are grouped into semantic slots so replacement only affects directly competing facts.**
- **Intent-aware retrieval can re-admit superseded facts for historical questions.**
- **The final context the LLM sees is built by ranking and packing eligible facts under a token budget.**

</div>

## 6. Source


Li, Y. & LI, Y. (2026). *What Should an Agent Forget? Separating What Is Stored from What Is Used*
